# 🟡 Solution: Flow Matching Loss (Rectified Flow)

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✅ SOLUTION

def flow_matching_loss(model, x0, x1, t):
    # x0: noise (B, ...)   x1: data (B, ...)   t: (B,) in [0, 1]

    # Reshape t to (B, 1, 1, ...) so it broadcasts over every feature dim
    t = t.view(-1, *([1] * (x1.dim() - 1)))

    # Straight-line probability path between noise and data
    x_t = (1.0 - t) * x0 + t * x1

    # Velocity of that path — constant in t
    target = x1 - x0

    # Regress the velocity field
    v = model(x_t, t)
    return ((v - target) ** 2).mean()

In [ ]:
# Verify
x0 = torch.randn(8, 4)
x1 = torch.randn(8, 4)
t = torch.rand(8)

print("zero-velocity model :", flow_matching_loss(lambda x, tt: torch.zeros_like(x), x0, x1, t).item())
print("expected            :", ((x1 - x0) ** 2).mean().item())
print("perfect model       :", flow_matching_loss(lambda x, tt: x1 - x0, x0, x1, t).item())

In [ ]:
from torch_judge import check
check("flow_matching")